# Choosing stack members and adding diagnosis pairs

This notebook extends the previous stacking approach by adding a few more diverse models and diagnosis-pair features. The same feature engineering and cross-validation setup is kept. The new models are added to capture different patterns in the data, while diagnosis pairs help the model learn relationships between conditions that appear together. Their predictions are then combined using the same stacking method to improve overall performance.

## Import Libraries and Load Dataset Paths

In [1]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

paths = ['/kaggle/input/competitions/machine-learning-fall-26-challenge-1-iba-karachi/',
         '/kaggle/input/machine-learning-fall-26-challenge-1-iba-karachi/',
         'kaggle/input/dataset/', '../kaggle/input/dataset/', '../../kaggle/input/dataset/', './']
DATA = next((Path(p) for p in paths if (Path(p) / 'train.csv').exists()), None)
if DATA is None:
    raise FileNotFoundError('Dataset not found. Check the dataset folder.')
OUT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('extra/outputs/boosting_progression/17_stacking_dx_pairs')
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
JOBS = 4
print('Data:', DATA)
print('Results:', OUT)

Data: /kaggle/input/competitions/machine-learning-fall-26-challenge-1-iba-karachi
Results: /kaggle/working


## Data Cleaning Helpers

In [2]:
# ---------------------------------------------------------------------------
# Cleaning helpers
# ---------------------------------------------------------------------------
def clean_text(series):
    s = series.astype('string').str.strip().str.upper()
    s = s.replace({'?': pd.NA, 'UNKNOWN': pd.NA, 'U': pd.NA, 'NOT AVAILABLE': pd.NA,
                   'NOT MAPPED': pd.NA, '': pd.NA, 'NAN': pd.NA})
    return s.astype(object).where(s.notna(), np.nan)


def clean_icd(code):
    code = str(code).strip().upper().replace('.', '')
    return code[:3] + '.' + code[3:] if len(code) > 3 else code


# Plausible range of each lab in the standard unit.  Values that land outside
# are first tried as a unit mix-up or a misplaced decimal before being dropped.
LAB_UNIT = {
    'Albumin': {'g/dL': 1, 'g/L': .1}, 'Hemoglobin': {'g/dL': 1, 'g/L': .1},
    'Creatinine': {'mg/dL': 1, 'umol/L': 1 / 88.4}, 'Glucose': {'mg/dL': 1, 'mmol/L': 18.0182},
    'Sodium': {'mmol/L': 1}, 'Potassium': {'mmol/L': 1}, 'Bun': {'mg/dL': 1},
    'Bnp': {'pg/mL': 1}, 'Hba1C': {'%': 1}, 'Inr': {'ratio': 1},
    'Lactate': {'mmol/L': 1}, 'Troponin_I': {'ng/mL': 1}, 'Wbc': {'10^3/uL': 1},
}
LAB_RANGE = {'Sodium': (110, 175), 'Potassium': (1.5, 9), 'Glucose': (20, 1200),
             'Creatinine': (.1, 20), 'Bun': (1, 200), 'Hemoglobin': (3, 22),
             'Wbc': (.2, 100), 'Albumin': (.8, 6.5), 'Lactate': (.1, 25),
             'Troponin_I': (0, 60), 'Bnp': (0, 40000), 'Hba1C': (3, 20), 'Inr': (.5, 12)}
# Reference ranges, used to fill the abnormal flag when it is blank.
LAB_NORMAL = {'Sodium': (135, 145), 'Potassium': (3.5, 5.1), 'Glucose': (70, 140),
              'Creatinine': (.6, 1.3), 'Bun': (7, 20), 'Hemoglobin': (12, 17.5),
              'Wbc': (4, 11), 'Albumin': (3.5, 5), 'Lactate': (.5, 2),
              'Troponin_I': (0, .04), 'Bnp': (0, 100), 'Hba1C': (4, 5.7), 'Inr': (.8, 1.2)}
VITAL_RANGE = {'HR': (25, 220), 'SBP': (50, 260), 'DBP': (25, 160), 'RR': (5, 60),
               'SPO2': (50, 100), 'TEMP': (33, 43)}


def standardise_labs(labs):
    value = pd.to_numeric(labs.value, errors='coerce')
    out = pd.Series(np.nan, index=labs.index)
    stats = {}
    for name, units in LAB_UNIT.items():
        rows = labs.lab_name.eq(name)
        low, high = LAB_RANGE[name]
        raw = value[rows]
        factor = labs.loc[rows, 'unit'].map(units)
        std = raw * factor
        # Blank unit: try every known unit and keep the one that is plausible.
        for scale in sorted(set(units.values())):
            fill = std.isna() & factor.isna() & (raw * scale).between(low, high)
            std[fill] = raw[fill] * scale
        # Out of range: a mislabelled unit or a value off by a factor of ten.
        bad = std.notna() & ~std.between(low, high)
        fixed = pd.Series(np.nan, index=std.index)
        for scale in [*sorted(set(units.values())), .1, 10]:
            trial = raw * scale
            ok = bad & fixed.isna() & trial.between(low, high)
            fixed[ok] = trial[ok]
        stats[name] = (int(bad.sum()), int(fixed.notna().sum()))
        std[bad] = fixed[bad]
        out[rows] = std
    return out, stats

## Feature Engineering

In [3]:
# ---------------------------------------------------------------------------
# Feature table
# ---------------------------------------------------------------------------
def build_features():
    t0 = time.time()
    read = lambda n: pd.read_csv(DATA / f'{n}.csv', low_memory=False).drop_duplicates()
    train, test = read('train'), read('test')
    enc, pat, hosp = read('index_encounters'), read('patients'), read('hospitals')
    dx, icd, proc = read('diagnoses'), read('icd10_lookup'), read('procedures')
    labs, vit, meds = read('lab_results'), read('vitals'), read('medications')
    tr, prior, plan = read('unit_transfers'), read('prior_utilization'), read('discharge_planning')

    for table, cols in [(pat, ['insurance_type', 'race_ethnicity', 'sex', 'language', 'marital_status']),
                        (enc, ['admission_type', 'admission_source', 'service_line',
                               'discharge_disposition', 'attending_specialty']),
                        (hosp, ['urban_rural', 'ownership'])]:
        for c in cols:
            table[c] = clean_text(table[c])
    enc['admit'] = pd.to_datetime(enc.admit_dt, format='mixed')
    enc['dis'] = pd.to_datetime(enc.discharge_dt, format='mixed')
    pat.loc[pat.birth_year.le(1900), 'birth_year'] = np.nan
    pat.loc[~pat.deprivation_index.between(0, 100), 'deprivation_index'] = np.nan
    plan['home_health_referral'] = clean_text(plan.home_health_referral).map({'Y': 1., 'N': 0.})
    plan['education_done'] = clean_text(plan.education_done).map({'YES': 1., 'NO': 0.})
    admit = enc.set_index('encounter_id').admit
    dis = enc.set_index('encounter_id').dis

    df = (enc.merge(pat, on='patient_id', how='left').merge(hosp, on='hospital_id', how='left')
          .merge(plan, on='encounter_id', how='left'))
    df = df.set_index('encounter_id')
    F = pd.DataFrame(index=df.index)
    F['age'] = df.admit.dt.year - df.birth_year
    F['los_days'] = (df.dis - df.admit).dt.total_seconds() / 86400
    F['los_nights'] = (df.dis.dt.normalize() - df.admit.dt.normalize()).dt.days
    F['admit_hour'] = df.admit.dt.hour
    F['discharge_hour'] = df.dis.dt.hour
    F['admit_dow'] = df.admit.dt.dayofweek
    F['discharge_dow'] = df.dis.dt.dayofweek
    F['discharge_weekend'] = (df.dis.dt.dayofweek >= 5).astype(int)
    F['admit_month'] = df.admit.dt.month
    F['admit_year'] = df.admit.dt.year
    for c in ['deprivation_index', 'bed_size', 'teaching_flag', 'admit_weight_kg', 'followup_scheduled_days',
              'home_health_referral', 'med_reconciliation_done', 'education_done']:
        F[c] = df[c]
    F['has_planning'] = df.index.isin(plan.encounter_id).astype(int)
    F['followup_missing'] = df.followup_scheduled_days.isna().astype(int)
    F['zip3_num'] = pd.to_numeric(df.zip3, errors='coerce')
    cats = ['hospital_id', 'attending_id', 'admission_type', 'admission_source', 'service_line',
            'discharge_disposition', 'attending_specialty', 'sex', 'race_ethnicity', 'language',
            'marital_status', 'insurance_type', 'urban_rural', 'ownership']
    for c in cats:
        F[c] = df[c]
    F['zip3'] = df.zip3.astype('Int64').astype(str).replace('<NA>', np.nan)
    F['specialty_matches_service'] = (df.attending_specialty.fillna('').str[:5]
                                      == df.service_line.fillna('').str[:5]).astype(int)

    # Diagnoses --------------------------------------------------------------
    dx['icd10_code'] = dx.icd10_code.map(clean_icd)
    dx = dx.drop_duplicates(['encounter_id', 'icd10_code'])
    dx = dx.merge(icd[['icd10_code', 'chapter', 'chronic_flag']], on='icd10_code', how='left')
    poa = dx.present_on_admission.astype(str).str.upper().str.strip()
    g = dx.groupby('encounter_id')
    F['n_dx'] = g.size()
    F['n_dx_chronic'] = g.chronic_flag.sum()
    F['n_dx_acute'] = F.n_dx - F.n_dx_chronic
    F['n_dx_chapters'] = g.chapter.nunique()
    for k in ['N', 'U', 'W']:
        F[f'n_dx_poa_{k}'] = dx[poa.eq(k)].groupby('encounter_id').size()
    F['n_dx_poa_blank'] = dx[poa.isin(['NAN', ''])].groupby('encounter_id').size()
    chap = pd.crosstab(dx.encounter_id, dx.chapter).add_prefix('chapter_')
    code = pd.crosstab(dx.encounter_id, dx.icd10_code).clip(upper=1)
    code = code[[c for c in code if code[c].sum() >= 30]].add_prefix('dx_')
    F = F.join(chap).join(code)
    prim = dx.sort_values('seq')[dx.is_primary == 1].drop_duplicates('encounter_id').set_index('encounter_id')
    F['primary_dx'] = prim.icd10_code
    F['primary_chapter'] = prim.chapter
    F['primary_chronic'] = prim.chronic_flag
    has = lambda *codes: code.reindex(F.index).filter(items=[f'dx_{c}' for c in codes]).sum(axis=1).clip(upper=1)
    F['cx_heart_failure'] = has('I50.9', 'I50.22', 'I50.23', 'I50.32', 'I50.33', 'I11.0')
    F['cx_ckd'] = has('N18.30', 'N18.4', 'N18.9', 'E11.22')
    F['cx_diabetes'] = has('E11.65', 'E11.10', 'E11.69', 'E11.9', 'E11.22')
    F['cx_copd'] = has('J44.1', 'J44.0', 'J44.9')
    F['cx_cancer'] = has('C34.90', 'C50.919', 'C18.9', 'C25.9', 'C80.1', 'C79.9')
    F['cx_metastatic'] = has('C79.9')
    F['cx_dementia'] = has('F03.90')
    F['cx_noncompliance'] = has('Z91.19')
    F['cx_count'] = F.filter(like='cx_').drop(columns='cx_metastatic').sum(axis=1)

    # Procedures ---------------------------------------------------------------
    proc['when'] = pd.to_datetime(proc.proc_dt, format='%d-%b-%Y', errors='coerce')
    F['n_procedures'] = proc.groupby('encounter_id').size()
    F = F.join(pd.crosstab(proc.encounter_id, proc.code.astype(str)).clip(upper=1).add_prefix('proc_'))
    days_to_proc = (proc.when - proc.encounter_id.map(admit).dt.normalize()).dt.days
    F['first_proc_day'] = days_to_proc.groupby(proc.encounter_id).min()
    F['days_proc_to_discharge'] = (proc.encounter_id.map(dis).dt.normalize() - proc.when).dt.days.groupby(proc.encounter_id).min()

    # Labs -----------------------------------------------------------------------
    labs['v'], lab_fix = standardise_labs(labs)
    labs['when'] = pd.to_datetime(labs.collected_dt, format='%Y-%m-%d %H:%M:%S')
    labs = labs.sort_values(['encounter_id', 'when'])
    flag = labs.abnormal_flag.astype(str).str.upper()
    lo = labs.lab_name.map({k: v[0] for k, v in LAB_NORMAL.items()})
    hi = labs.lab_name.map({k: v[1] for k, v in LAB_NORMAL.items()})
    labs['high'] = np.where(flag.eq('H'), 1, np.where(flag.isin(['N', 'L']), 0, (labs.v > hi).astype(int)))
    labs['low'] = np.where(flag.eq('L'), 1, np.where(flag.isin(['N', 'H']), 0, (labs.v < lo).astype(int)))
    labs['hours_before_discharge'] = (labs.encounter_id.map(dis) - labs.when).dt.total_seconds() / 3600
    g = labs.groupby(['encounter_id', 'lab_name'])
    st = g.v.agg(['first', 'last', 'min', 'max', 'mean', 'std', 'count']).unstack()
    st.columns = [f'lab_{lab}_{s}' for s, lab in st.columns]
    F = F.join(st)
    for lab in LAB_UNIT:
        if f'lab_{lab}_last' in F:
            F[f'lab_{lab}_delta'] = F[f'lab_{lab}_last'] - F[f'lab_{lab}_first']
    ab = g[['high', 'low']].sum().unstack()
    ab.columns = [f'lab_{lab}_n_{s}' for s, lab in ab.columns]
    F = F.join(ab)
    last = labs.groupby(['encounter_id', 'lab_name'])[['high', 'low']].last().sum(axis=1).unstack()
    F['labs_last_abnormal'] = last.sum(axis=1)
    F['n_labs'] = labs.groupby('encounter_id').size()
    F['n_labs_abnormal'] = labs.groupby('encounter_id')[['high', 'low']].sum().sum(axis=1)
    F['n_labs_flag_blank'] = labs[flag.isin(['NAN'])].groupby('encounter_id').size()
    F['lab_hours_last_before_discharge'] = labs.groupby('encounter_id').hours_before_discharge.min()
    F['n_lab_types'] = labs.groupby('encounter_id').lab_name.nunique()
    # Severity: how far each reading sits outside its reference range, in
    # range widths (log scale for the skewed markers).  Summed over tests.
    skew = {'Bnp', 'Troponin_I', 'Creatinine', 'Lactate', 'Bun', 'Wbc', 'Inr'}
    width = (hi - lo).where(~labs.lab_name.isin(skew), np.log1p(hi) - np.log1p(lo))
    val = labs.v.where(~labs.lab_name.isin(skew), np.log1p(labs.v))
    l2 = lo.where(~labs.lab_name.isin(skew), np.log1p(lo))
    h2 = hi.where(~labs.lab_name.isin(skew), np.log1p(hi))
    labs['dev'] = ((l2 - val).clip(lower=0) + (val - h2).clip(lower=0)) / width
    g = labs.groupby(['encounter_id', 'lab_name']).dev
    for s in ['first', 'last', 'max', 'mean']:
        per = g.agg(s).unstack()
        F[f'lab_sev_{s}'] = per.sum(axis=1)
        F[f'lab_n_abn_{s}'] = per.gt(0).sum(axis=1)
    F['lab_sev_improve'] = F.lab_sev_first - F.lab_sev_last
    given = labs[flag.isin(['H', 'L'])].groupby('encounter_id')
    F['n_labs_flag_H'] = labs[flag.eq('H')].groupby('encounter_id').size()
    F['n_labs_flag_L'] = labs[flag.eq('L')].groupby('encounter_id').size()
    F['n_lab_types_flagged'] = given.lab_name.nunique()

    # Vitals ---------------------------------------------------------------------
    vit['value'] = pd.to_numeric(vit.value, errors='coerce')
    f_temp = vit.vital_type.eq('TEMP') & vit.value.between(80, 115)
    vit.loc[f_temp, 'value'] = (vit.loc[f_temp, 'value'] - 32) / 1.8
    for name, (low, high) in VITAL_RANGE.items():
        bad = vit.vital_type.eq(name) & ~vit.value.between(low, high)
        vit.loc[bad, 'value'] = np.nan
    vit['when'] = pd.to_datetime(vit.charttime, format='%m/%d/%Y %H:%M')
    vit = vit.sort_values(['encounter_id', 'when'])
    vit['hb'] = (vit.encounter_id.map(dis) - vit.when).dt.total_seconds() / 3600
    g = vit.groupby(['encounter_id', 'vital_type']).value
    st = g.agg(['first', 'last', 'min', 'max', 'mean', 'std', 'count']).unstack()
    st.columns = [f'vital_{v}_{s}' for s, v in st.columns]
    F = F.join(st)
    late = vit[vit.hb <= 24].groupby(['encounter_id', 'vital_type']).value.agg(['mean', 'max', 'min']).unstack()
    late.columns = [f'vital_{v}_last24h_{s}' for s, v in late.columns]
    F = F.join(late)
    v = vit.value
    t = vit.vital_type
    flags = pd.DataFrame({'encounter_id': vit.encounter_id,
                          'n_tachycardia': t.eq('HR') & (v > 100), 'n_bradycardia': t.eq('HR') & (v < 50),
                          'n_hypotension': t.eq('SBP') & (v < 90), 'n_hypertension': t.eq('SBP') & (v > 180),
                          'n_hypoxia': t.eq('SPO2') & (v < 90), 'n_fever': t.eq('TEMP') & (v > 38),
                          'n_tachypnea': t.eq('RR') & (v > 22)})
    F = F.join(flags.groupby('encounter_id').sum())
    vnorm = {'HR': (60, 100), 'SBP': (100, 140), 'DBP': (60, 90), 'RR': (12, 20), 'SPO2': (94, 100), 'TEMP': (36.1, 37.8)}
    vlo, vhi = t.map({k: a for k, (a, b) in vnorm.items()}), t.map({k: b for k, (a, b) in vnorm.items()})
    vit['dev'] = ((vlo - v).clip(lower=0) + (v - vhi).clip(lower=0)) / (vhi - vlo)
    g = vit.groupby(['encounter_id', 'vital_type']).dev
    for s in ['first', 'last', 'max', 'mean']:
        F[f'vital_sev_{s}'] = g.agg(s).unstack().sum(axis=1)
    F['n_vitals'] = vit.groupby('encounter_id').size()
    F['vital_hours_last_before_discharge'] = vit.groupby('encounter_id').hb.min()
    # Instability on the final reading of each vital (early-warning style score).
    lv = F
    F['unstable_at_discharge'] = ((lv.vital_HR_last > 100).astype(int) + (lv.vital_SBP_last < 90).astype(int)
                                  + (lv.vital_SPO2_last < 92).astype(int) + (lv.vital_TEMP_last > 37.8).astype(int)
                                  + (lv.vital_RR_last > 22).astype(int))

    # Medications ----------------------------------------------------------------
    meds['drug_class'] = meds.drug_class.astype(str).str.upper().str.strip()
    meds['route'] = meds.route.astype(str).str.upper().str.strip()
    meds['start'] = pd.to_datetime(meds.start_dt.astype('Int64').astype(str), format='%Y%m%d', errors='coerce')
    meds['stop'] = pd.to_datetime(meds.stop_dt.astype('Int64').astype(str), format='%Y%m%d', errors='coerce')
    cont = meds.continued_at_discharge.eq(1)
    F['n_meds'] = meds.groupby('encounter_id').size()
    F['n_meds_continued'] = meds[cont].groupby('encounter_id').size()
    F['n_meds_continued_blank'] = meds[meds.continued_at_discharge.isna()].groupby('encounter_id').size()
    F['n_meds_stopped'] = meds[meds.stop.notna()].groupby('encounter_id').size()
    F['n_drug_classes'] = meds.groupby('encounter_id').drug_class.nunique()
    F['n_classes_continued'] = meds[cont].groupby('encounter_id').drug_class.nunique()
    F['days_med_start_after_admit'] = (meds.start - meds.encounter_id.map(admit).dt.normalize()).dt.days.groupby(meds.encounter_id).max()
    F['n_meds_started_last_day'] = meds[meds.start.dt.normalize().eq(meds.encounter_id.map(dis).dt.normalize())].groupby('encounter_id').size()
    F = F.join(pd.crosstab(meds.encounter_id, meds.drug_class).add_prefix('med_'))
    F = F.join(pd.crosstab(meds[cont].encounter_id, meds[cont].drug_class).add_prefix('med_continued_'))
    F = F.join(pd.crosstab(meds.encounter_id, meds.route).drop(columns=['INH'], errors='ignore').add_prefix('route_'))
    high_risk = ['ANTICOAGULANT', 'INSULIN', 'OPIOID', 'ANTIPSYCHOTIC', 'BENZODIAZEPINE', 'LOOP_DIURETIC']
    F['n_high_risk_continued'] = meds[cont & meds.drug_class.isin(high_risk)].groupby('encounter_id').drug_class.nunique()

    # Unit transfers ---------------------------------------------------------------
    tr['in'] = pd.to_datetime(tr.in_dt, format='mixed')
    tr['out'] = pd.to_datetime(tr.out_dt, format='mixed')
    tr['hours'] = (tr.out - tr['in']).dt.total_seconds().clip(lower=0) / 3600
    tr['unit'] = tr.unit.astype(str).str.upper().str.strip()
    tr = tr.sort_values(['encounter_id', 'in'])
    F = F.join(tr.pivot_table(index='encounter_id', columns='unit', values='hours', aggfunc='sum').add_prefix('hours_'))
    F['n_transfers'] = tr.groupby('encounter_id').size()
    F['n_icu_visits'] = tr[tr.unit.eq('ICU')].groupby('encounter_id').size()
    F['last_unit'] = tr.groupby('encounter_id').unit.last()
    F['first_unit'] = tr.groupby('encounter_id').unit.first()
    seq = tr.groupby('encounter_id').unit.agg(lambda s: '>'.join(s))
    F['icu_bounce'] = seq.str.contains(r'(WARD|STEPDOWN).*ICU').astype(int)
    F['hours_icu_fraction'] = F.get('hours_ICU', 0) / (F.los_days * 24).replace(0, np.nan)

    # Prior utilisation ----------------------------------------------------------------
    pa = enc.set_index('patient_id')
    prior['days'] = (prior.patient_id.map(pa.admit).dt.normalize() - pd.to_datetime(prior.visit_dt)).dt.days
    prior = prior[prior.days > 0]
    prior['same_hosp'] = prior.hospital_id.eq(prior.patient_id.map(pa.hospital_id))
    P = pd.DataFrame(index=pa.index)
    for w in [30, 90, 180, 365, 730]:
        r = prior[prior.days <= w]
        P = P.join(pd.crosstab(r.patient_id, r.visit_type).add_prefix('prior_').add_suffix(f'_{w}d'))
    P = P.join(prior.groupby(['patient_id', 'visit_type']).days.min().unstack().add_prefix('days_since_'))
    P['days_since_any'] = prior.groupby('patient_id').days.min()
    P['prior_hospitals'] = prior.groupby('patient_id').hospital_id.nunique()
    P['prior_same_hospital'] = prior[prior.same_hosp].groupby('patient_id').size()
    for kind in ['ED', 'IP', 'OP']:
        r = prior[prior.visit_type.eq(kind)]
        P[f'prior_{kind}_decay90'] = np.exp(-r.days / 90).groupby(r.patient_id).sum()
    count_like = [c for c in P if c.startswith('prior_')]
    P[count_like] = P[count_like].fillna(0)
    F = F.join(P.reindex(df.patient_id).set_axis(df.index))
    F['prior_acute_365d'] = F.prior_ED_365d + F.prior_IP_365d

    # Ratios and simple risk scores ----------------------------------------------------------
    F['labs_per_day'] = F.n_labs / F.los_days.clip(lower=.25)
    F['meds_per_day'] = F.n_meds / F.los_days.clip(lower=.25)
    F['abnormal_lab_fraction'] = F.n_labs_abnormal / F.n_labs
    # LACE-style index: length of stay, acuity, comorbidity, ED visits.
    los_pts = pd.cut(F.los_days, [-1, 1, 2, 3, 6, 13, 1e9], labels=[0, 1, 2, 3, 4, 5]).astype(float)
    F['lace'] = (los_pts + 3 * df.admission_type.eq('EMERGENCY').astype(int)
                 + F.n_dx_chronic.fillna(0).clip(upper=5) + F.prior_ED_180d.clip(upper=4))

    F = add_unsupervised(F, df)

    zero = [c for c in F if c.startswith(('chapter_', 'dx_', 'proc_', 'med_', 'route_', 'hours_', 'n_'))
            and c not in ['n_dx'] and F[c].dtype != object]
    F[zero] = F[zero].fillna(0)
    F['n_dx'] = F.n_dx.fillna(0)
    F = F.replace([np.inf, -np.inf], np.nan)
    cat_cols = [c for c in F if F[c].dtype == object]
    for c in cat_cols:
        F[c] = F[c].astype(str).replace('nan', np.nan)

    y = train.set_index('encounter_id').readmit_30d
    meta = {'cat_cols': cat_cols, 'lab_fixes': lab_fix, 'n_features': F.shape[1]}
    F.to_pickle(OUT / 'features.pkl')
    y.to_pickle(OUT / 'y.pkl')
    test[['encounter_id']].to_pickle(OUT / 'test_ids.pkl')
    (OUT / 'features_meta.json').write_text(json.dumps(meta, indent=2))
    print(f'features {F.shape}, cats {len(cat_cols)}, built in {time.time() - t0:.0f}s')
    print('lab values out of range / repaired:', lab_fix)
    return F


def add_unsupervised(F, df):
    """Label-free features fitted on train and test stays together.

    1. Latent severity: the labs, vitals and counts are noisy views of one
       hidden severity.  Factor analysis / PCA on those columns estimates it
       more cleanly than any single column.
    2. Hospital case mix: each hospital's average patient, and how far this
       patient sits from it.
    No readmission labels are used, so fitting on all stays does not leak.
    """
    from sklearn.decomposition import FactorAnalysis, PCA
    from sklearn.preprocessing import QuantileTransformer
    stat_cols = [c for c in F if (c.startswith(('lab_', 'vital_')) and
                                  c.rsplit('_', 1)[-1] in ('first', 'last', 'min', 'max', 'mean')
                                  and '_sev_' not in c)
                 and F[c].notna().mean() > .6]
    count_cols = ['lab_sev_first', 'lab_sev_last', 'lab_sev_max', 'lab_sev_mean', 'vital_sev_max', 'vital_sev_mean',
                  'n_labs_abnormal', 'labs_last_abnormal', 'abnormal_lab_fraction', 'n_dx', 'n_dx_chronic',
                  'cx_count', 'age', 'los_days', 'prior_ED_365d', 'prior_IP_365d', 'prior_OP_730d',
                  'n_meds', 'n_drug_classes', 'deprivation_index']
    cols = list(dict.fromkeys(c for c in stat_cols + count_cols if c in F))
    Z = F[cols].astype(float)
    Z = Z.fillna(Z.median())
    Z = QuantileTransformer(n_quantiles=500, output_distribution='normal', random_state=SEED).fit_transform(Z)
    fa = FactorAnalysis(n_components=6, random_state=SEED).fit_transform(Z)
    pc = PCA(n_components=6, random_state=SEED).fit_transform(Z)
    for i in range(fa.shape[1]):
        F[f'latent_fa{i}'] = fa[:, i]
        F[f'latent_pc{i}'] = pc[:, i]
    lab_only = [c for c in stat_cols if c.startswith('lab_')]
    Zl = F[lab_only].astype(float)
    Zl = QuantileTransformer(n_quantiles=500, output_distribution='normal',
                             random_state=SEED).fit_transform(Zl.fillna(Zl.median()))
    F['latent_lab_fa'] = FactorAnalysis(n_components=1, random_state=SEED).fit_transform(Zl)[:, 0]

    hosp = df.hospital_id
    for c in ['age', 'lab_sev_mean', 'n_dx_chronic', 'cx_count', 'los_days', 'prior_acute_365d' if 'prior_acute_365d' in F else 'prior_ED_365d',
              'latent_fa0', 'deprivation_index']:
        m = F[c].groupby(hosp).transform('mean')
        F[f'hosp_mean_{c}'] = m
        F[f'hosp_diff_{c}'] = F[c] - m
    F['hosp_volume'] = hosp.map(hosp.value_counts())
    att = df.attending_id
    F['att_mean_latent_fa0'] = F.latent_fa0.groupby(att).transform('mean')
    F['att_volume'] = att.map(att.value_counts())
    return F


def load():
    F = pd.read_pickle(OUT / 'features.pkl')
    y = pd.read_pickle(OUT / 'y.pkl')
    test_ids = pd.read_pickle(OUT / 'test_ids.pkl').encounter_id
    cats = json.loads((OUT / 'features_meta.json').read_text())['cat_cols']
    return F, y, test_ids, cats

## Boosting and Logistic Models (with DART, GOSS and No-Vitals LightGBM)

In [4]:
# ---------------------------------------------------------------------------
# Models
# ---------------------------------------------------------------------------
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

HIGH_CARD = ['attending_id', 'zip3', 'primary_dx']   # target-encoded inside each fold


def target_encode(fit_col, fit_y, apply_cols, smooth=50, seed=SEED):
    """Smoothed mean target. Training rows get out-of-fold values."""
    prior = fit_y.mean()

    def table(col, target):
        s = pd.DataFrame({'c': col.values, 'y': target.values}).groupby('c').y.agg(['sum', 'count'])
        return (s['sum'] + prior * smooth) / (s['count'] + smooth)

    oof = pd.Series(np.nan, index=fit_col.index)
    for a, b in StratifiedKFold(5, shuffle=True, random_state=seed).split(fit_col, fit_y):
        t = table(fit_col.iloc[a], fit_y.iloc[a])
        oof.iloc[b] = fit_col.iloc[b].map(t).astype(float).values
    t = table(fit_col, fit_y)
    return oof.fillna(prior), [c.map(t).astype(float).fillna(prior) for c in apply_cols]


def prepare(Xf, yf, Xo, cats, spec):
    """Fold-local preprocessing: drop columns, target-encode, set category dtypes."""
    drop = [c for c in spec.get('drop', []) if c in Xf]
    drop += [c for c in Xf if spec.get('drop_prefix') and c.startswith(tuple(spec['drop_prefix']))]
    Xf, Xo = Xf.drop(columns=drop), [x.drop(columns=drop) for x in Xo]
    cats = [c for c in cats if c in Xf]
    te = [c for c in spec.get('te', []) if c in Xf]
    for c in te:
        a, bs = target_encode(Xf[c], yf, [x[c] for x in Xo], smooth=spec.get('te_smooth', 50))
        Xf[f'te_{c}'] = a.values
        for x, b in zip(Xo, bs):
            x[f'te_{c}'] = b.values
    removed = [c for c in spec.get('drop_raw', te) if c in Xf]
    Xf, Xo = Xf.drop(columns=removed), [x.drop(columns=removed) for x in Xo]
    cats = [c for c in cats if c in Xf]
    if spec.get('select'):
        # Keep the top-N columns by gain from a quick booster fitted on this fold only.
        import lightgbm as lgb
        z = Xf.copy()
        for c in cats:
            z[c] = z[c].astype('category')
        m = lgb.LGBMClassifier(**{**LGB_BASE, 'n_estimators': 800, 'learning_rate': .02},
                               importance_type='gain', verbosity=-1, n_jobs=JOBS, random_state=SEED).fit(z, yf)
        keep = pd.Series(m.feature_importances_, Xf.columns).sort_values(ascending=False).index[:spec['select']]
        Xf, Xo = Xf[keep], [x[keep] for x in Xo]
        cats = [c for c in cats if c in Xf]
    return Xf, Xo, cats


def logistic_init(Xf, yf, Xo, cats, inner_folds=5):
    """Spline-logistic log-odds used as LightGBM's starting point (init_score).

    Training rows get out-of-fold log-odds from an inner split of the training
    fold; validation/test rows get a logistic fitted on the whole training fold.
    So no row's own label ever feeds its starting score.
    """
    import non_boost as NB
    from scipy.special import logit
    spec = NB.MODELS['nb_spline_l1']
    inner = np.zeros(len(Xf))
    for a, b in StratifiedKFold(inner_folds, shuffle=True, random_state=SEED).split(Xf, yf):
        inner[b] = NB.fit_predict(spec, Xf.iloc[a], yf.iloc[a], [Xf.iloc[b]], cats)[0]
    outer = NB.fit_predict(spec, Xf, yf, Xo, cats)
    to_logit = lambda q: logit(np.clip(q, 1e-5, 1 - 1e-5))
    return to_logit(inner), [to_logit(q) for q in outer]


def fit_predict(spec, Xf, yf, Xo, cats):
    family = spec['family']
    init_fit = init_out = None
    if spec.get('init_logistic'):
        init_fit, init_out = logistic_init(Xf, yf, Xo, cats)
    Xf, Xo, cats = prepare(Xf.copy(), yf, [x.copy() for x in Xo], cats, spec)
    p = dict(spec.get('params', {}))
    seeds = spec.get('seeds', [SEED])
    outs = [np.zeros(len(x)) for x in Xo]
    for seed in seeds:
        if family == 'lgb':
            import lightgbm as lgb
            levels = {c: pd.Index(Xf[c].dropna().unique()) for c in cats}
            def to_cat(x):
                x = x.copy()
                for c in cats:
                    x[c] = pd.Categorical(x[c], categories=levels[c])
                return x
            if spec.get('monotone'):
                p = {**p, 'monotone_constraints': monotone_vector(Xf.columns),
                     'monotone_constraints_method': spec.get('monotone_method', 'advanced')}
            m = lgb.LGBMClassifier(verbosity=-1, n_jobs=JOBS, random_state=seed, **p)
            if init_fit is not None:
                # Trees learn only what the logistic start misses; scores = start + trees.
                m.fit(to_cat(Xf), yf, init_score=init_fit)
                preds = [1 / (1 + np.exp(-(m.predict(to_cat(x), raw_score=True) + i)))
                         for x, i in zip(Xo, init_out)]
            else:
                m.fit(to_cat(Xf), yf)
                preds = [m.predict_proba(to_cat(x))[:, 1] for x in Xo]
        elif family == 'xgb':
            import xgboost as xgb
            def enc(x):
                x = x.copy()
                for c in cats:
                    x[c] = pd.Categorical(x[c], categories=pd.Index(Xf[c].dropna().unique()))
                return x
            m = xgb.XGBClassifier(tree_method='hist', enable_categorical=True, n_jobs=JOBS,
                                  random_state=seed, eval_metric='auc', **p)
            m.fit(enc(Xf), yf)
            preds = [m.predict_proba(enc(x))[:, 1] for x in Xo]
        elif family == 'cat':
            from catboost import CatBoostClassifier
            def enc(x):
                x = x.copy()
                for c in cats:
                    x[c] = x[c].fillna('MISSING').astype(str)
                return x
            m = CatBoostClassifier(verbose=False, thread_count=JOBS, random_seed=seed,
                                   allow_writing_files=False, cat_features=cats, **p)
            m.fit(enc(Xf), yf)
            preds = [m.predict_proba(enc(x))[:, 1] for x in Xo]
        elif family == 'logit':
            from sklearn.linear_model import LogisticRegression
            from sklearn.preprocessing import OneHotEncoder, StandardScaler, QuantileTransformer
            from sklearn.impute import SimpleImputer
            from sklearn.compose import ColumnTransformer
            from sklearn.pipeline import make_pipeline
            num = [c for c in Xf if c not in cats]
            if spec.get('log'):
                # Skewed non-negative columns (counts, hours, markers) on a log scale
                # and every numeric column clipped to its fitting-fold 0.5-99.5% range.
                skewed = [c for c in num if Xf[c].min() >= 0 and Xf[c].skew() > 2]
                q = Xf[num].quantile([.005, .995])
                def tf(x):
                    x = x.copy()
                    x[num] = x[num].clip(q.iloc[0], q.iloc[1], axis=1)
                    x[skewed] = np.log1p(x[skewed])
                    return x
                Xf, Xo = tf(Xf), [tf(x) for x in Xo]
            scaler = (QuantileTransformer(n_quantiles=200, output_distribution='normal')
                      if spec.get('quantile') else StandardScaler())
            ct = ColumnTransformer([
                ('n', make_pipeline(SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True), scaler), num),
                ('c', make_pipeline(SimpleImputer(strategy='constant', fill_value='MISSING'),
                                    OneHotEncoder(handle_unknown='ignore', min_frequency=20)), cats)])
            m = make_pipeline(ct, LogisticRegression(max_iter=3000, solver='liblinear', **p))
            m.fit(Xf, yf)
            preds = [m.predict_proba(x)[:, 1] for x in Xo]
        else:
            raise ValueError(family)
        outs = [o + q / len(seeds) for o, q in zip(outs, preds)]
        if family == 'logit':
            break
    return outs


def run_cv(name, spec, F, y, test_ids, cats, save=True, folds=5, verbose=True, fold_seed=SEED):
    X = F.loc[y.index]
    Xt = F.loc[test_ids]
    oof = pd.Series(np.nan, index=y.index)
    tp = np.zeros(len(Xt))
    scores = []
    t0 = time.time()
    for k, (a, b) in enumerate(StratifiedKFold(folds, shuffle=True, random_state=fold_seed).split(X, y)):
        pv, pt = fit_predict(spec, X.iloc[a], y.iloc[a], [X.iloc[b], Xt], cats)
        oof.iloc[b] = pv
        tp += pt / folds
        scores.append(roc_auc_score(y.iloc[b], pv))
    mean = float(np.mean(scores))
    if verbose:
        print(f'{name:45s} AUC {mean:.5f} ±{np.std(scores):.4f} | pooled {roc_auc_score(y, oof):.5f} '
              f'| {time.time() - t0:.0f}s | {[round(s, 4) for s in scores]}', flush=True)
    if save:
        (OUT / 'preds').mkdir(exist_ok=True)
        pd.to_pickle({'oof': oof, 'test': pd.Series(tp, index=test_ids.values), 'auc': mean,
                      'folds': scores, 'spec': spec}, OUT / 'preds' / f'{name}.pkl')
    return mean, oof, tp


def fold_auc(y, p):
    return float(np.mean([roc_auc_score(y.iloc[b], p.iloc[b])
                          for _, b in StratifiedKFold(5, shuffle=True, random_state=SEED).split(p, y)]))


LGB_BASE = {'n_estimators': 2000, 'learning_rate': .01, 'num_leaves': 7, 'min_child_samples': 300,
            'reg_lambda': 30, 'colsample_bytree': .2, 'subsample': .8, 'subsample_freq': 1,
            'cat_smooth': 50, 'min_data_per_group': 200, 'max_cat_to_onehot': 8}

# Monotone directions (risk only rises / only falls).  Checked against the data:
# per-lab high/low counts disagree for low values, so they stay unconstrained.
MONO_UP = [r'^n_dx$', r'^n_dx_chronic$', r'^cx_(?!metastatic)', r'^n_labs_abnormal$', r'^labs_last_abnormal$',
           r'^abnormal_lab_fraction$', r'^lab_sev_(first|last|max|mean)$', r'^lab_n_abn_', r'^vital_sev_',
           r'^n_(hypoxia|tachycardia|hypotension|fever|tachypnea)$', r'^unstable_at_discharge$', r'^prior_(ED|IP)_\d+d$',
           r'^prior_acute_365d$', r'^prior_(ED|IP)_decay90$', r'^lace$', r'^age$', r'^n_high_risk_continued$',
           r'^n_icu_visits$', r'^prior_hospitals$']
MONO_DOWN = [r'^days_since_(ED|IP|any)$']


def monotone_vector(columns):
    import re
    return [1 if any(re.match(p, c) for p in MONO_UP) else -1 if any(re.match(p, c) for p in MONO_DOWN) else 0
            for c in columns]


NO_SIGNAL = ['attending_id', 'zip3']
# Exact copies of another column (or constant): four chapters hold one code each, three
# comorbidity flags cover one code, HbA1c is measured once per stay, etc.
DUPLICATES = ['dx_D64.9', 'dx_W19.XXXA', 'dx_A41.9', 'dx_S72.001A', 'cx_metastatic', 'cx_dementia',
              'cx_noncompliance', 'lab_Hba1C_last', 'lab_Hba1C_min', 'lab_Hba1C_max', 'lab_Hba1C_mean',
              'lab_n_abn_mean', 'lab_Hba1C_std', 'last_unit']        # tested: no gain, native categories overfit
LGB4 = {**LGB_BASE, 'num_leaves': 4, 'n_estimators': 3500, 'learning_rate': .008}
MODELS = {
    'lgb_base': {'family': 'lgb', 'params': LGB_BASE, 'drop': NO_SIGNAL},
    'lgb4': {'family': 'lgb', 'params': LGB4, 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'seeds': [42, 7, 2024]},
    'lgb4_top200': {'family': 'lgb', 'params': LGB4, 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'select': 200},
    # Latent-severity / hospital case-mix columns lower LightGBM (0.75285 -> 0.7522-0.7526), so dropped here;
    # they help the logistic model slightly (0.74798 -> 0.74821) and stay there.
    'lgb4_top120': {'family': 'lgb', 'params': LGB4, 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'select': 120,
                    'drop_prefix': ['latent_', 'hosp_', 'att_'],
                    'seeds': [42, 7, 2024]},
    'lgb4_top80': {'family': 'lgb', 'params': LGB4, 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'select': 80,
                   'seeds': [42, 7, 2024]},
    # Optuna (60 trials on fold seed 7, scripts/reach_075.py tune lgb), hardcoded.
    # Alone on seed-42 folds: 0.75276 (current lgb4_top120: 0.75286); adds variety to the blend.
    'lgb_tuned': {'family': 'lgb', 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'select': 120,
                  'drop_prefix': ['latent_', 'hosp_', 'att_'], 'seeds': [42, 7, 2024],
                  'params': {**LGB_BASE, 'learning_rate': 0.0052, 'num_leaves': 3, 'min_child_samples': 192, 'reg_lambda': 3.94858, 'reg_alpha': 1.05565, 'colsample_bytree': 0.30974, 'subsample': 0.68768, 'min_split_gain': 0.24768, 'path_smooth': 72.98701, 'max_bin': 255, 'cat_smooth': 175.44344, 'n_estimators': 5403}},
    'lgb4_mono': {'family': 'lgb', 'params': LGB4, 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'select': 120,
                  'drop_prefix': ['latent_', 'hosp_', 'att_'], 'seeds': [42, 7, 2024], 'monotone': True},
    # Linear trees: every leaf fits a small linear model on the features in its path.
    'lgb4_linear': {'family': 'lgb', 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'select': 120,
                    'drop_prefix': ['latent_', 'hosp_', 'att_'],
                    'params': {**LGB4, 'linear_tree': True, 'linear_lambda': 5.0}},
    'lgb4_linear_l50': {'family': 'lgb', 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'select': 120,
                        'drop_prefix': ['latent_', 'hosp_', 'att_'],
                        'params': {**LGB4, 'linear_tree': True, 'linear_lambda': 50.0}},
    # Boost on top of the spline logistic: trees only learn what logistic misses.
    # Fewer rounds / slower rate because the start is already good.
    'lgb4_on_logit': {'family': 'lgb', 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'select': 120,
                      'drop_prefix': ['latent_', 'hosp_', 'att_'], 'init_logistic': True,
                      'params': {**LGB4, 'n_estimators': 2000, 'learning_rate': .006}},
    'lgb7': {'family': 'lgb', 'params': {**LGB_BASE, 'learning_rate': .006, 'n_estimators': 3000},
             'drop': NO_SIGNAL, 'te': ['primary_dx'], 'seeds': [11, 12]},
    'xgb3': {'family': 'xgb', 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'seeds': [42, 5],
             'params': {'n_estimators': 2500, 'learning_rate': .01, 'max_depth': 3, 'min_child_weight': 60,
                        'reg_lambda': 40, 'subsample': .8, 'colsample_bytree': .25, 'max_cat_to_onehot': 1}},
    'cat5': {'family': 'cat', 'drop': NO_SIGNAL, 'seeds': [42],
             'params': {'iterations': 3000, 'learning_rate': .03, 'depth': 5, 'l2_leaf_reg': 20,
                        'rsm': .3, 'border_count': 128}},
    'logit_l1': {'family': 'logit', 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'quantile': True,
                 'params': {'penalty': 'l1', 'C': .05}},
    'logit_log': {'family': 'logit', 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'log': True,
                  'params': {'penalty': 'l1', 'C': .05}},
    'logit_log_l2': {'family': 'logit', 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'log': True,
                     'params': {'penalty': 'l2', 'C': .005}},
    'logit_std': {'family': 'logit', 'drop': NO_SIGNAL, 'te': ['primary_dx'],
                  'params': {'penalty': 'l1', 'C': .1}},
}


# Extra base models for stacking variety (they make different mistakes from lgb4_top120).
MODELS['lgb_dart'] = {'family': 'lgb', 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'select': 120,
                      'drop_prefix': ['latent_', 'hosp_', 'att_'], 'seeds': [42],
                      'params': {**LGB_BASE, 'boosting_type': 'dart', 'num_leaves': 4, 'n_estimators': 1200,
                                 'learning_rate': .05, 'drop_rate': .1, 'skip_drop': .5}}
MODELS['lgb_nolab'] = {'family': 'lgb', 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'seeds': [42, 7],
                       'drop_prefix': ['latent_', 'hosp_', 'att_', 'lab_', 'labs_', 'n_labs', 'abnormal_lab'],
                       'params': LGB4}


# Kept by the forward search over eight candidate boosters.
_STACK = {'drop': NO_SIGNAL, 'te': ['primary_dx'], 'select': 120, 'drop_prefix': ['latent_', 'hosp_', 'att_']}
MODELS['lgb_dart2'] = {'family': 'lgb', **_STACK, 'seeds': [7],
                       'params': {**LGB_BASE, 'boosting_type': 'dart', 'num_leaves': 7, 'n_estimators': 1000,
                                  'learning_rate': .05, 'drop_rate': .15, 'skip_drop': .5, 'colsample_bytree': .3}}
MODELS['lgb_goss'] = {'family': 'lgb', **_STACK, 'seeds': [42, 7],
                      'params': {**LGB4, 'data_sample_strategy': 'goss', 'subsample': 1.0, 'subsample_freq': 0}}
MODELS['lgb_novitals'] = {'family': 'lgb', 'drop': NO_SIGNAL, 'te': ['primary_dx'], 'seeds': [42, 7],
                          'drop_prefix': ['latent_', 'hosp_', 'att_', 'vital_', 'n_vitals', 'n_tachy', 'n_brady',
                                          'n_hypo', 'n_hyper', 'n_fever', 'unstable_'],
                          'params': LGB4}

## Other Models (Spline Logistic, Random Forest, KNN)

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, SplineTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

DROP = ['attending_id', 'zip3']
SPLINE_TOP = 40          # strongest numeric columns that get spline terms


def log_clip(Xf, Xo, num):
    """Clip to the fitting fold's 0.5-99.5% range; log skewed non-negative columns."""
    skewed = [c for c in num if Xf[c].min() >= 0 and Xf[c].skew() > 2]
    q = Xf[num].quantile([.005, .995])

    def tf(x):
        x = x.copy()
        x[num] = x[num].clip(q.iloc[0], q.iloc[1], axis=1)
        x[skewed] = np.log1p(x[skewed])
        return x
    return tf(Xf), [tf(x) for x in Xo]


def strongest(Xf, yf, num, k):
    """Top-k numeric columns by univariate AUC distance from 0.5, on the fitting fold."""
    score = {}
    for c in num:
        v = Xf[c]
        if v.nunique() > 10:
            score[c] = abs(roc_auc_score(yf, v.fillna(v.median())) - .5)
    return sorted(score, key=score.get, reverse=True)[:k]


def other_fit_predict(spec, Xf, yf, Xo, cats):
    """Fold-local preprocessing + one non-boosting model."""
    Xf, Xo, cats = prepare(Xf.copy(), yf, [x.copy() for x in Xo], cats,
                             {'drop': DROP, 'te': ['primary_dx'], 'drop_prefix': spec.get('drop_prefix', []),
                              'select': spec.get('select')})
    num = [c for c in Xf if c not in cats]
    kind = spec['kind']
    if kind in ('logit', 'spline'):
        Xf, Xo = log_clip(Xf, Xo, num)
        parts = [('n', make_pipeline(SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True),
                                     StandardScaler()), num),
                 ('c', make_pipeline(SimpleImputer(strategy='constant', fill_value='MISSING'),
                                     OneHotEncoder(handle_unknown='ignore', min_frequency=20)), cats)]
        if kind == 'spline':
            top = strongest(Xf, yf, num, spec.get('top', SPLINE_TOP))
            parts.append(('s', make_pipeline(SimpleImputer(strategy='median'),
                                             QuantileTransformer(n_quantiles=200, random_state=SEED),
                                             SplineTransformer(n_knots=spec.get('knots', 6), degree=3)), top))
        m = make_pipeline(ColumnTransformer(parts),
                          LogisticRegression(penalty=spec.get('penalty', 'l1'), C=spec['C'], solver='liblinear',
                                             max_iter=4000))
    elif kind in ('rf', 'et', 'tree'):
        # Trees take ordinal codes for categories and the raw numbers (NaN -> -999).
        enc = {c: {v: i for i, v in enumerate(Xf[c].dropna().unique())} for c in cats}

        def tf(x):
            x = x.copy()
            for c in cats:
                x[c] = x[c].map(enc[c]).fillna(-1)
            return x.fillna(-999).astype(np.float32)
        Xf, Xo = tf(Xf), [tf(x) for x in Xo]
        cls = {'rf': RandomForestClassifier, 'et': ExtraTreesClassifier}.get(kind)
        m = (DecisionTreeClassifier(random_state=SEED, **spec['params']) if kind == 'tree'
             else cls(n_jobs=JOBS, random_state=SEED, **spec['params']))
    elif kind in ('knn', 'nb'):
        cols = [c for c in num if c.startswith(spec['prefix'])] if spec.get('prefix') else strongest(Xf, yf, num, spec['top'])
        m = make_pipeline(SimpleImputer(strategy='median'),
                          QuantileTransformer(n_quantiles=200, output_distribution='normal', random_state=SEED),
                          KNeighborsClassifier(n_neighbors=spec['k'], weights='distance', n_jobs=JOBS)
                          if kind == 'knn' else GaussianNB())
        Xf, Xo = Xf[cols], [x[cols] for x in Xo]
    else:
        raise ValueError(kind)
    m.fit(Xf, yf)
    return [m.predict_proba(x)[:, 1] for x in Xo]


OTHER_MODELS = {
    'nb_spline_l1': {'kind': 'spline', 'C': .05},
    'nb_rf': {'kind': 'rf', 'params': {'n_estimators': 600, 'min_samples_leaf': 40, 'max_features': .15,
                                       'max_samples': .7}},
    'nb_knn_top30': {'kind': 'knn', 'top': 30, 'k': 400},
}


def run_other_cv(name, spec):
    """Same five folds as run_cv, for the non-boosting models; saves out-of-fold and test predictions."""
    X, Xt = F.loc[y.index], F.loc[test_ids]
    oof, tp, scores = pd.Series(np.nan, index=y.index), np.zeros(len(Xt)), []
    for fit_rows, check_rows in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y):
        held, test_part = other_fit_predict(spec, X.iloc[fit_rows], y.iloc[fit_rows], [X.iloc[check_rows], Xt], cats)
        oof.iloc[check_rows] = held
        tp += test_part / 5
        scores.append(roc_auc_score(y.iloc[check_rows], held))
    print(f'{name:22s} ROC-AUC {np.mean(scores):.5f}', flush=True)
    (OUT / 'preds').mkdir(exist_ok=True)
    pd.to_pickle({'oof': oof, 'test': pd.Series(tp, index=test_ids.values), 'auc': float(np.mean(scores))},
                 OUT / 'preds' / f'{name}.pkl')

## Stacking

In [6]:
from scipy.special import logit

BOOSTERS = ['lgb4_top120', 'lgb_tuned', 'lgb4_linear', 'xgb3', 'cat5', 'lgb_dart', 'lgb_nolab',
            'lgb_goss', 'lgb_dart2', 'lgb_novitals']
LINEAR = ['logit_log']
OTHERS = list(OTHER_MODELS)
PAIRS = ['lgb4_dxpairs']
STACKED = BOOSTERS + LINEAR + OTHERS + PAIRS


def stack_inputs(predictions, rows):
    """Log-odds of each model's prediction plus one-hot discharge disposition."""
    inputs = pd.DataFrame({name: logit(np.clip(values, 1e-6, 1 - 1e-6)) for name, values in predictions.items()},
                          index=rows)
    disposition = pd.get_dummies(F.loc[rows, 'discharge_disposition'].fillna('NA'), prefix='discharge_disposition').astype(float)
    return inputs.join(disposition)


def stack_and_submit():
    saved = {name: pd.read_pickle(OUT / 'preds' / f'{name}.pkl') for name in STACKED}
    sample = pd.read_csv(DATA / 'sample_submission.csv')
    ids = sample.encounter_id
    train_inputs = stack_inputs({n: saved[n]['oof'].loc[y.index].values for n in STACKED}, y.index)
    test_inputs = stack_inputs({n: saved[n]['test'].loc[ids].values for n in STACKED}, ids)
    test_inputs = test_inputs.reindex(columns=train_inputs.columns, fill_value=0)
    mean, spread = train_inputs.mean(), train_inputs.std().replace(0, 1)
    train_inputs, test_inputs = (train_inputs - mean) / spread, (test_inputs - mean) / spread

    # Score the stacker with the same five folds, on out-of-fold predictions only.
    checked = pd.Series(np.nan, index=y.index)
    for fit_rows, check_rows in StratifiedKFold(5, shuffle=True, random_state=SEED).split(train_inputs, y):
        stacker = LogisticRegression(max_iter=3000).fit(train_inputs.iloc[fit_rows], y.iloc[fit_rows])
        checked.iloc[check_rows] = stacker.predict_proba(train_inputs.iloc[check_rows])[:, 1]
    print(f'Stacking ROC-AUC: {fold_auc(y, checked):.5f}')

    stacker = LogisticRegression(max_iter=3000).fit(train_inputs, y)
    display(pd.Series(stacker.coef_[0], index=train_inputs.columns).round(3).to_frame('weight'))
    submission = pd.DataFrame({'encounter_id': ids, 'readmit_30d': stacker.predict_proba(test_inputs)[:, 1]})
    assert submission.encounter_id.equals(ids) and submission.readmit_30d.between(0, 1).all()
    submission.to_csv(OUT / 'submission.csv', index=False)
    print('Saved: submission.csv', submission.shape)

## Diagnosis Pair Features

In [7]:
from itertools import combinations


def read_table(name):
    return pd.read_csv(DATA / f'{name}.csv', low_memory=False).drop_duplicates()


READ = read_table


def dxpair_features(min_count=400):
    dx = READ('diagnoses')
    icd = READ('icd10_lookup')
    dx['icd10_code'] = dx.icd10_code.map(clean_icd)
    dx = dx.drop_duplicates(['encounter_id', 'icd10_code']).merge(icd[['icd10_code', 'chapter']], on='icd10_code', how='left')
    D = pd.DataFrame(index=READ('index_encounters').encounter_id)
    for level, col in [('chap', 'chapter'), ('code', 'icd10_code')]:
        sets = dx.dropna(subset=[col]).groupby('encounter_id')[col].agg(lambda s: sorted(set(s)))
        pairs = sets.map(lambda xs: list(combinations(xs, 2))).explode().dropna()
        counts = pairs.value_counts()
        common = counts[counts >= min_count].index
        hits = pairs[pairs.isin(common)]
        flags = pd.crosstab(hits.index, hits.map(lambda p: f'pair_{level}_{p[0]}__{p[1]}')).clip(upper=1)
        D = D.join(flags)
        D[f'pair_{level}_count'] = pairs.groupby(level=0).size()
    return D.fillna(0)


# Same booster as lgb4_top120, fitted on the table with the diagnosis-pair flags added.
MODELS['lgb4_dxpairs'] = {**MODELS['lgb4_top120'], 'seeds': [42, 7, 2024]}

## Train Models and Create Submission

In [8]:
from IPython.display import display

F = build_features()
F, y, test_ids, cats = load()
for name in BOOSTERS + LINEAR:
    run_cv(name, MODELS[name], F, y, test_ids, cats)
for name, spec in OTHER_MODELS.items():
    run_other_cv(name, spec)
pairs = dxpair_features()
print('Diagnosis-pair columns:', pairs.shape[1])
run_cv('lgb4_dxpairs', MODELS['lgb4_dxpairs'], F.join(pairs), y, test_ids, cats)
stack_and_submit()

features (109787, 522), cats 19, built in 144s
lab values out of range / repaired: {'Albumin': (19, 15), 'Hemoglobin': (90, 59), 'Creatinine': (19, 17), 'Glucose': (72, 72), 'Sodium': (114, 114), 'Potassium': (94, 94), 'Bun': (51, 51), 'Bnp': (0, 0), 'Hba1C': (6, 6), 'Inr': (3, 3), 'Lactate': (10, 10), 'Troponin_I': (0, 0), 'Wbc': (44, 44)}
lgb4_top120                                   AUC 0.75264 ±0.0061 | pooled 0.75254 | 403s | [np.float64(0.7555), np.float64(0.7598), np.float64(0.7503), np.float64(0.7556), np.float64(0.742)]
lgb_tuned                                     AUC 0.75279 ±0.0061 | pooled 0.75270 | 599s | [np.float64(0.755), np.float64(0.7597), np.float64(0.7506), np.float64(0.7566), np.float64(0.7421)]
lgb4_linear                                   AUC 0.75243 ±0.0060 | pooled 0.75234 | 248s | [np.float64(0.7554), np.float64(0.7595), np.float64(0.7496), np.float64(0.7556), np.float64(0.7421)]
xgb3                                          AUC 0.75111 ±0.0063 | pooled 0.751

,weight
lgb4_top120,-0.175
lgb_tuned,0.204
lgb4_linear,0.402
xgb3,-0.142
cat5,-0.064
lgb_dart,0.283
lgb_nolab,0.071
lgb_goss,-0.549
lgb_dart2,0.222
lgb_novitals,0.187


Saved: submission.csv (32812, 2)
